# Edge AI Assignment 3 — Quantization & Pruning (ResNet-18, CIFAR-10)

## Question 1 — PTQ & QAT

In [1]:
# ==================== Q1 · Cell 1: baseline, PTQ, QAT ====================

# ---------- [M0] settings ----------
import copy, pickle, random, subprocess, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.ao.quantization import get_default_qconfig, get_default_qat_qconfig, prepare, prepare_qat, convert, disable_observer
from torch.ao.nn.intrinsic.qat import freeze_bn_stats
from torchvision.models.quantization import resnet18
warnings.filterwarnings("ignore")

QUICK = False                                     # True = smoke test on a tiny subset
EPOCHS, QAT_EPOCHS, BS = (1, 1, 256) if QUICK else (25, 3, 256)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
SAVE, RES = Path("saved_models"), Path("results")
SAVE.mkdir(exist_ok=True); RES.mkdir(exist_ok=True)
torch.backends.quantized.engine = "fbgemm"
torch.manual_seed(1); random.seed(1)

# ---------- [M1] data: read the raw CIFAR-10 batch files into tensors ----------
ROOT = next(p for p in ["../data/cifar-10-batches-py", "data/cifar-10-batches-py"] if Path(p).exists())
MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)

def read(files):
    batches = [pickle.load(open(f"{ROOT}/{f}", "rb"), encoding="bytes") for f in files]
    x = torch.tensor(np.concatenate([b[b"data"] for b in batches])).view(-1, 3, 32, 32) / 255
    return (x - MEAN) / STD, torch.tensor(sum([b[b"labels"] for b in batches], []))

X_train, y_train = read([f"data_batch_{i}" for i in range(1, 6)])
X_test, y_test = read(["test_batch"])
LABELS = [n.decode() for n in pickle.load(open(f"{ROOT}/batches.meta", "rb"), encoding="bytes")[b"label_names"]]
if QUICK:
    X_train, y_train, X_test, y_test = X_train[:2000], y_train[:2000], X_test[:1000], y_test[:1000]
Xg, yg = X_train.to(DEV), y_train.to(DEV)                  # training set kept on the GPU

# ---------- [M2] model ----------
def build():
    net = resnet18(weights=None, quantize=False, num_classes=10)
    net.conv1, net.maxpool = nn.Conv2d(3, 64, 3, padding=1, bias=False), nn.Identity()
    return net

# ---------- [M3] train / test helpers ----------
def test(net, device=DEV):
    net.eval().to(device)
    with torch.no_grad():
        pred = torch.cat([net(x.to(device)).argmax(1).cpu() for x in X_test.split(1000)])
    return round((pred == y_test).float().mean().item() * 100, 2), pred

def fit(net, epochs, lr, before_epoch=None):
    net.to(DEV)
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=5e-4)
    steps = len(Xg) // BS
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=epochs * steps)
    for ep in range(epochs):
        if before_epoch: before_epoch(net, ep)
        net.train()
        order = torch.randperm(len(Xg), device=DEV)
        for i in range(steps):
            idx = order[i * BS:(i + 1) * BS]
            x = torch.where(torch.rand(len(idx), 1, 1, 1, device=DEV) < 0.5, Xg[idx].flip(3), Xg[idx])  # random flip
            x = torch.roll(x, (random.randint(-4, 4), random.randint(-4, 4)), (2, 3))                    # random shift
            loss = F.cross_entropy(net(x), yg[idx], label_smoothing=0.1)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        print(f"  epoch {ep + 1}/{epochs}: loss {loss.item():.3f}, test acc {test(net)[0]}%")

# ---------- [M4] Q1.1 baseline FP32 model ----------
model_fp32 = build()
if (SAVE / "fp32_weights.pth").exists() and not QUICK:
    model_fp32.load_state_dict(torch.load(SAVE / "fp32_weights.pth"))
else:
    fit(model_fp32, EPOCHS, 0.1)
    torch.save(model_fp32.state_dict(), SAVE / "fp32_weights.pth")
print("FP32 accuracy:", test(model_fp32)[0])

# ---------- [M5] Q1.2 post-training static quantization ----------
def ptq(n_calib):
    net = copy.deepcopy(model_fp32).cpu().eval()
    net.fuse_model()                                       # fuse conv + bn + relu
    net.qconfig = get_default_qconfig("fbgemm")            # x86 INT8 backend
    net = prepare(net)                                     # insert observers
    with torch.no_grad():
        for x in X_train[torch.randperm(len(X_train))[:n_calib]].split(100):   # calibration on training images
            net(x)
    return convert(net)                                    # FP32 -> INT8

# calibration-size sweep before fixing the setting
ptq_sweep = {n: ptq(n) for n in [100, 1000, 5000]}
for n, net in ptq_sweep.items():
    print(f"PTQ with {n} calibration images: accuracy {test(net, 'cpu')[0]}%")
model_ptq = ptq_sweep[1000]                                # 1000 images used for the final PTQ model
torch.save(model_ptq.state_dict(), SAVE / "ptq_int8_weights.pth")

# ---------- [M6] Q1.3 quantization-aware training ----------
model_qat = copy.deepcopy(model_fp32).cpu().train()        # start from the pretrained FP32 model
model_qat.fuse_model(is_qat=True)
model_qat.qconfig = get_default_qat_qconfig("fbgemm")
model_qat = prepare_qat(model_qat)                         # insert fake-quantization modules

def qat_schedule(net, ep):
    if ep >= 1: net.apply(freeze_bn_stats)                 # freeze BatchNorm statistics
    if ep >= 2: net.apply(disable_observer)                # freeze scales / zero-points

fit(model_qat, QAT_EPOCHS, 1e-3, qat_schedule)             # QAT fine-tuning
model_qat = convert(model_qat.cpu().eval())                # INT8 model
torch.save(model_qat.state_dict(), SAVE / "qat_int8_weights.pth")
print("QAT accuracy:", test(model_qat, "cpu")[0])

  epoch 1/25: loss 1.620, test acc 42.85%
  epoch 2/25: loss 1.283, test acc 58.18%
  epoch 3/25: loss 1.216, test acc 69.56%
  epoch 4/25: loss 1.210, test acc 70.86%
  epoch 5/25: loss 1.066, test acc 69.78%
  epoch 6/25: loss 0.920, test acc 76.32%
  epoch 7/25: loss 0.977, test acc 75.97%
  epoch 8/25: loss 0.893, test acc 82.18%
  epoch 9/25: loss 0.847, test acc 85.27%
  epoch 10/25: loss 0.803, test acc 78.55%
  epoch 11/25: loss 0.748, test acc 81.18%
  epoch 12/25: loss 0.735, test acc 84.39%
  epoch 13/25: loss 0.746, test acc 84.03%
  epoch 14/25: loss 0.717, test acc 83.37%
  epoch 15/25: loss 0.646, test acc 84.21%
  epoch 16/25: loss 0.676, test acc 86.75%
  epoch 17/25: loss 0.627, test acc 88.11%
  epoch 18/25: loss 0.632, test acc 87.94%
  epoch 19/25: loss 0.654, test acc 88.47%
  epoch 20/25: loss 0.577, test acc 89.87%
  epoch 21/25: loss 0.604, test acc 91.72%
  epoch 22/25: loss 0.558, test acc 92.05%
  epoch 23/25: loss 0.545, test acc 92.83%
  epoch 24/25: loss 

In [2]:
# ==================== Q1 · Cell 2: inference, benchmark, ONNX ====================
import onnxruntime as ort
from torch.utils.benchmark import Timer

# ---------- [M7] Q1.4 TorchScript export + inference pipeline ----------
sample = X_test[:1]
for name, net in {"fp32": model_fp32, "ptq_int8": model_ptq, "qat_int8": model_qat}.items():
    torch.jit.save(torch.jit.trace(net.cpu().eval(), sample), SAVE / f"{name}.ts")

def load(name):
    return torch.jit.load(SAVE / f"{name}.ts").eval()

def classify(name, images):
    with torch.no_grad():
        return [LABELS[i] for i in load(name)(images).argmax(1)]

print(pd.DataFrame({"true": [LABELS[i] for i in y_test[:5]]} | {n: classify(n, X_test[:5]) for n in ["fp32", "ptq_int8", "qat_int8"]}))

# ---------- [M8] Q1.5 benchmark on the test set (CPU) ----------
torch.set_num_threads(4)

def bench(names, tag):
    rows, preds = [], pd.DataFrame({"label": y_test.numpy()})
    for name in names:
        net = load(name)
        t = time.perf_counter()
        acc, pred = test(net, "cpu")
        t = time.perf_counter() - t
        latency = Timer("with torch.no_grad(): net(x)", globals={"net": net, "x": sample, "torch": torch}, num_threads=4).timeit(100).mean
        rows.append(dict(model=name, top1=acc, latency_ms=latency * 1000, images_per_s=len(y_test) / t,
                         file_mb=(SAVE / f"{name}.ts").stat().st_size / 1e6))
        preds[name] = pred.numpy()
    preds.to_csv(RES / f"{tag}_predictions.csv")
    return pd.DataFrame(rows)

q1 = bench(["fp32", "ptq_int8", "qat_int8"], "q1")
q1.to_csv(RES / "q1_benchmark.csv", index=False)
display(q1)

# ---------- [M9] ONNX export + ONNX Runtime CPU inference ----------
ONNX_ROWS = []
def to_onnx(name, net):
    path = f"{SAVE}/{name}.onnx"
    torch.onnx.export(net.cpu().eval(), sample, path, input_names=["x"], output_names=["y"],
                      dynamic_axes={"x": {0: "n"}, "y": {0: "n"}}, opset_version=17, dynamo=False)
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    logits = np.concatenate([sess.run(None, {"x": c})[0] for c in np.array_split(X_test.numpy(), 10)])
    ONNX_ROWS.append(dict(model=name, onnx_top1=round(float((logits.argmax(1) == y_test.numpy()).mean()) * 100, 2),
                          onnx_mb=Path(path).stat().st_size / 1e6))
    print(ONNX_ROWS[-1])

for name, net in {"fp32": model_fp32, "ptq_int8": model_ptq, "qat_int8": model_qat}.items():
    to_onnx(name, net)

       true      fp32  ptq_int8  qat_int8
0       cat       cat       cat       cat
1      ship      ship      ship      ship
2      ship      ship      ship      ship
3  airplane  airplane  airplane  airplane
4      frog      frog      frog      frog


,model,top1,latency_ms,images_per_s,file_mb
0,fp32,92.96,6.694098,225.873247,44.901244
1,ptq_int8,92.82,2.373711,586.422177,11.437522
2,qat_int8,92.91,2.799838,593.884629,11.440740


{'model': 'fp32', 'onnx_top1': 92.96, 'onnx_mb': 44.68592}
{'model': 'ptq_int8', 'onnx_top1': 92.93, 'onnx_mb': 11.294103}
{'model': 'qat_int8', 'onnx_top1': 93.03, 'onnx_mb': 11.294103}


## Question 2 — Pruning

In [3]:
# ==================== Q2 · Cell 1: unstructured + structured pruning ====================
import torch.nn.utils.prune as prune

# ---------- [M10] Q2.1 pretrained ResNet-18 (from Q1) ----------
original = copy.deepcopy(model_fp32)
nonzero = lambda net: sum(int((p != 0).sum()) for p in net.parameters())

# analytic cost model: parameters and MACs of conv / linear layers
def conv_cost(h, w, c_in, c_out, k, stride=1, groups=1, bias=True):
    oh, ow = -(-h // stride), -(-w // stride)
    params = (k * k * (c_in // groups) * c_out) + (c_out if bias else 0)
    macs = oh * ow * c_out * (k * k * (c_in // groups))
    return params, macs, oh, ow

def dense_cost(n_in, n_out):
    return n_in * n_out + n_out, n_in * n_out

def macs(net):
    layers, sizes = [m for m in net.modules() if isinstance(m, (nn.Conv2d, nn.Linear))], {}
    hooks = [m.register_forward_hook(lambda m, i, o: sizes.__setitem__(m, i[0].shape[-1])) for m in layers]
    with torch.no_grad():
        net.cpu().eval()(X_test[:1])
    [h.remove() for h in hooks]
    return sum(conv_cost(sizes[m], sizes[m], m.in_channels, m.out_channels, m.kernel_size[0], m.stride[0])[1]
               if isinstance(m, nn.Conv2d) else dense_cost(m.in_features, m.out_features)[1] for m in layers)
print("original:", test(original)[0], "% |", nonzero(original), "non-zero params |", macs(original), "MACs")

# ---------- [M11] Q2.2 unstructured: layer-wise L1 magnitude pruning, 3 rounds (~75%) ----------
model_unstruct = copy.deepcopy(original)
targets = [m for m in model_unstruct.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
for r in range(3):
    for m in targets:
        prune.l1_unstructured(m, "weight", amount=0.37)   # masks 37% of the remaining weights of each layer
    print(f"round {r + 1}: accuracy after pruning {test(model_unstruct)[0]}% -> fine-tuning")
    fit(model_unstruct, 1 if QUICK else 2, 0.01)           # Q2.4 fine-tuning
for m in targets:
    prune.remove(m, "weight")                             # make pruning permanent
sparsity = 1 - sum(int((m.weight != 0).sum()) for m in targets) / sum(m.weight.numel() for m in targets)
print(f"unstructured: sparsity {sparsity:.1%}, accuracy {test(model_unstruct)[0]}%")

# ---------- [M12] Q2.3 structured: remove 40% of each block's inner channels (smallest BN |gamma|) ----------
model_struct = copy.deepcopy(original).cpu()
for layer in [model_struct.layer1, model_struct.layer2, model_struct.layer3, model_struct.layer4]:
    for blk in layer:
        n_keep = int(blk.bn1.num_features * 0.6)
        keep = blk.bn1.weight.detach().abs().topk(n_keep).indices
        new_conv1 = nn.Conv2d(blk.conv1.in_channels, n_keep, 3, blk.conv1.stride, 1, bias=False)
        new_bn1 = nn.BatchNorm2d(n_keep)
        new_conv2 = nn.Conv2d(n_keep, blk.conv2.out_channels, 3, 1, 1, bias=False)
        with torch.no_grad():
            new_conv1.weight.copy_(blk.conv1.weight[keep])         # keep selected filters
            new_conv2.weight.copy_(blk.conv2.weight[:, keep])      # matching input channels of the next conv
            for attr in ["weight", "bias", "running_mean", "running_var"]:
                getattr(new_bn1, attr).copy_(getattr(blk.bn1, attr)[keep])
        blk.conv1, blk.bn1, blk.conv2 = new_conv1, new_bn1, new_conv2
print(f"structured: params {nonzero(original)} -> {nonzero(model_struct)}, accuracy before fine-tuning {test(model_struct)[0]}%")
fit(model_struct, 1 if QUICK else 6, 0.01)                 # Q2.4 fine-tuning
print("structured accuracy after fine-tuning:", test(model_struct)[0])

original: 92.95 % | 11173962 non-zero params | 555422720 MACs
round 1: accuracy after pruning 91.78% -> fine-tuning
  epoch 1/2: loss 0.533, test acc 92.89%
  epoch 2/2: loss 0.532, test acc 92.81%
round 2: accuracy after pruning 78.99% -> fine-tuning
  epoch 1/2: loss 0.556, test acc 92.71%
  epoch 2/2: loss 0.544, test acc 92.78%
round 3: accuracy after pruning 36.49% -> fine-tuning
  epoch 1/2: loss 0.610, test acc 91.97%
  epoch 2/2: loss 0.544, test acc 92.56%
unstructured: sparsity 75.0%, accuracy 92.56%
structured: params 11173962 -> 6766018, accuracy before fine-tuning 40.9%
  epoch 1/6: loss 0.594, test acc 90.9%
  epoch 2/6: loss 0.565, test acc 91.27%
  epoch 3/6: loss 0.583, test acc 91.77%
  epoch 4/6: loss 0.544, test acc 91.73%
  epoch 5/6: loss 0.530, test acc 92.23%
  epoch 6/6: loss 0.525, test acc 92.45%
structured accuracy after fine-tuning: 92.45


In [4]:
# ==================== Q2 · Cell 2: export, inference, benchmark ====================

# ---------- [M13] Q2.5 export the pruned models ----------
torch.save(model_unstruct.state_dict(), SAVE / "unstructured_weights.pth")
torch.save(model_struct, SAVE / "structured_full_model.pth")      # whole model, includes the new layer sizes
for name, net in {"unstructured": model_unstruct, "structured": model_struct}.items():
    torch.jit.save(torch.jit.trace(net.cpu().eval(), sample), SAVE / f"{name}.ts")
    to_onnx(name, net)
pd.DataFrame(ONNX_ROWS).to_csv(RES / "onnx_results.csv", index=False)

# ---------- [M14] inference for all five models + standalone ONNX CPU script ----------
ALL = ["fp32", "ptq_int8", "qat_int8", "unstructured", "structured"]
print(pd.DataFrame({"true": [LABELS[i] for i in y_test[:5]]} | {n: classify(n, X_test[:5]) for n in ALL}))

Path("run_inference.py").write_text("""# usage: python run_inference.py <fp32|ptq_int8|qat_int8|unstructured|structured> [n_images]
import sys, pickle, numpy as np, onnxruntime as ort
from pathlib import Path
name, n = sys.argv[1], int(sys.argv[2]) if len(sys.argv) > 2 else 1000
root = "../data/cifar-10-batches-py" if Path("../data/cifar-10-batches-py").exists() else "data/cifar-10-batches-py"
d = pickle.load(open(f"{root}/test_batch", "rb"), encoding="bytes")
x = d[b"data"][:n].reshape(-1, 3, 32, 32) / 255
x = (x - np.array([0.4914, 0.4822, 0.4465]).reshape(1, 3, 1, 1)) / np.array([0.2470, 0.2435, 0.2616]).reshape(1, 3, 1, 1)
pred = ort.InferenceSession(f"saved_models/{name}.onnx").run(None, {"x": x.astype(np.float32)})[0].argmax(1)
print(f"{name}: accuracy on {n} test images = {(pred == np.array(d[b'labels'][:n])).mean() * 100:.2f}%")
""")
for name in ALL:
    print(subprocess.run([sys.executable, "run_inference.py", name, "1000"], capture_output=True, text=True).stdout.strip())

# ---------- [M15] Q2.6 benchmark: original vs unstructured vs structured ----------
q2 = bench(["fp32", "unstructured", "structured"], "q2")
q2["nonzero_params"] = [nonzero(n) for n in (original, model_unstruct, model_struct)]
q2["macs"] = [macs(n) for n in (original, model_unstruct, model_struct)]
q2.to_csv(RES / "q2_benchmark.csv", index=False)
display(q2)

# ---------- final ranking of all five models: MnasNet-style reward (accuracy vs. model size) ----------
def reward(acc, params, param_budget, w=-0.07):
    return acc * ((params / param_budget) ** w)

final = pd.concat([q1, q2[q2.model != "fp32"]], ignore_index=True)
final["reward"] = reward(final.top1, final.file_mb, param_budget=12)   # cost = model size, budget = 12 MB edge device
final.to_csv(RES / "all_models.csv", index=False)
display(final.sort_values("reward", ascending=False))

{'model': 'unstructured', 'onnx_top1': 92.56, 'onnx_mb': 44.68592}
{'model': 'structured', 'onnx_top1': 92.43, 'onnx_mb': 27.05722}
       true      fp32  ptq_int8  qat_int8 unstructured structured
0       cat       cat       cat       cat          cat        cat
1      ship      ship      ship      ship         ship       ship
2      ship      ship      ship      ship         ship       ship
3  airplane  airplane  airplane  airplane     airplane   airplane
4      frog      frog      frog      frog         frog       frog
fp32: accuracy on 1000 test images = 93.90%
ptq_int8: accuracy on 1000 test images = 93.60%
qat_int8: accuracy on 1000 test images = 93.90%
unstructured: accuracy on 1000 test images = 93.40%
structured: accuracy on 1000 test images = 93.00%


,model,top1,latency_ms,images_per_s,file_mb,nonzero_params,macs
0,fp32,92.96,6.679459,223.583755,44.901244,11173962,555422720
1,unstructured,92.56,7.024035,224.971174,44.917188,2801225,555422720
2,structured,92.43,4.782474,301.590794,27.277674,6766018,334349312


,model,top1,latency_ms,images_per_s,file_mb,nonzero_params,macs,reward
2,qat_int8,92.91,2.799838,593.884629,11.440740,NaN,NaN,93.220915
1,ptq_int8,92.82,2.373711,586.422177,11.437522,NaN,NaN,93.132447
4,structured,92.43,4.782474,301.590794,27.277674,6766018.0,334349312.0,87.266815
0,fp32,92.96,6.694098,225.873247,44.901244,NaN,NaN,84.758003
3,unstructured,92.56,7.024035,224.971174,44.917188,2801225.0,555422720.0,84.391198
